In [42]:
"""Metadata filtering module for Research Paper Library Search System.

This module provides functionality to filter and rank research papers based on
metadata signals such as topic matching, freshness (publication date), quality
(citation count), and department. It combines semantic similarity scores with
metadata signals for improved ranking.
"""

from datetime import datetime
from typing import Dict, List, Optional

import numpy as np
import pandas as pd


class MetadataFilter:
    """
    Filters and ranks research papers based on metadata signals and relevance criteria.

    This class implements intelligent filtering and ranking strategies that consider:
    - Topic matching (e.g., "machine learning" queries should return ML papers)
    - Publication freshness (recent papers ranked higher)
    - Quality indicators (papers with higher citation counts ranked higher)
    - Department filtering (e.g., computer science vs mathematics)

    Attributes:
        topic_weight (float): Weight for topic matching (0.0–1.0).
        freshness_weight (float): Weight for publication freshness (0.0–1.0).
        quality_weight (float): Weight for citation-based quality (0.0–1.0).
        similarity_weight (float): Weight for semantic similarity (0.0–1.0).
    """

    def __init__(
        self,
        topic_weight: float = 0.2,
        freshness_weight: float = 0.3,
        quality_weight: float = 0.2,
        similarity_weight: float = 0.3
    ):
        """
        Initializes the MetadataFilter with configurable weights.

        Args:
            topic_weight (float): Weight for topic matching (default: 0.2).
            freshness_weight (float): Weight for publication recency (default: 0.3).
            quality_weight (float): Weight for citation-based quality (default: 0.2).
            similarity_weight (float): Weight for semantic similarity (default: 0.3).

        Raises:
            TypeError: If any weight is not a float.
            ValueError: If any weight is not in range [0.0, 1.0] or weights don't sum to ~1.0.
        """
        if not isinstance(topic_weight, (int, float)):
            raise TypeError("topic_weight must be a float")
        if not isinstance(freshness_weight, (int, float)):
            raise TypeError("freshness_weight must be a float")
        if not isinstance(quality_weight, (int, float)):
            raise TypeError("quality_weight must be a float")
        if not isinstance(similarity_weight, (int, float)):
            raise TypeError("similarity_weight must be a float")

        if not 0.0 <= topic_weight <= 1.0:
            raise ValueError("topic_weight must be between 0.0 and 1.0")
        if not 0.0 <= freshness_weight <= 1.0:
            raise ValueError("freshness_weight must be between 0.0 and 1.0")
        if not 0.0 <= quality_weight <= 1.0:
            raise ValueError("quality_weight must be between 0.0 and 1.0")
        if not 0.0 <= similarity_weight <= 1.0:
            raise ValueError("similarity_weight must be between 0.0 and 1.0")

        # if not np.isclose(
        #     topic_weight + freshness_weight + quality_weight + similarity_weight,
        #     1.0,
        # ):
        #     raise ValueError("Weights must sum to 1.0")

        self.topic_weight = float(topic_weight)
        self.freshness_weight = float(freshness_weight)
        self.quality_weight = float(quality_weight)
        self.similarity_weight = float(similarity_weight)

    def filter_by_metadata(
        self,
        df: pd.DataFrame,
        topic: Optional[str] = None,
        author_department: Optional[str] = None
    ) -> pd.DataFrame:
        """
        Filters papers by metadata criteria (topic and/or department).

        Args:
            df (pd.DataFrame): DataFrame with paper data containing:
                - 'topic': Research topic
                - 'author_department': Author department
            topic (Optional[str]): Topic to filter by (e.g., 'machine-learning').
            author_department (Optional[str]): Department to filter by (e.g., 'computer-science').

        Returns:
            pd.DataFrame: Filtered DataFrame with papers matching the criteria.

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            TypeError: If inputs have incorrect types.
        """
        required_columns = ['topic', 'author_department']

        if df is None or not isinstance(df, pd.DataFrame):
            raise TypeError("df must be a pandas DataFrame")

        if df.empty or not all(col in df.columns for col in required_columns):
            raise ValueError("DataFrame is empty or missing required columns.")

        if not pd.api.types.is_string_dtype(df["topic"]):
            raise TypeError("topic column must contain strings.")

        if not pd.api.types.is_string_dtype(df["author_department"]):
            raise TypeError("author_department column must contain strings.")

        if not isinstance(topic, (str, type(None))):
            raise TypeError("topic must be a string or None")

        if not isinstance(author_department, (str, type(None))):
            raise TypeError("author_department must be a string or None")

        if topic is not None:
            df = df[df['topic'].str.lower() == topic.lower()]

        if author_department is not None:
            df = df[df['author_department'].str.lower() == author_department.lower()]

        if df.empty:
            raise ValueError("No papers found matching the criteria.")

        return df

    def calculate_freshness_score(
        self,
        publication_date: str,
        min_date: datetime,
        max_date: datetime
    ) -> float:
        """
        Calculates a freshness score based on a paper's publication date.

        More recent papers receive higher scores. Score is normalized between 0.0 and 1.0.

        Args:
            publication_date (str): Date string in 'YYYY-MM-DD' format.
            min_date (datetime): Earliest date in the dataset.
            max_date (datetime): Latest date in the dataset.

        Returns:
            float: Freshness score between 0.0 and 1.0 (higher = more recent).
                 Returns 0.0 if date cannot be parsed or is invalid.

        Raises:
            ValueError: If date string cannot be parsed (should return 0.0 as fallback).
        """
        try:
            date = datetime.strptime(publication_date, "%Y-%m-%d")
        except ValueError:
            return 0.0

        if not (min_date <= date <= max_date):
            return 0.0

        # Normalize score to [0.0, 1.0]
        if max_date == min_date:
            return 1.0

        return (
            (date - min_date).total_seconds()
            / (max_date - min_date).total_seconds()
        )

    def calculate_quality_score(
        self,
        citation_count: int,
        min_citations: int,
        max_citations: int
    ) -> float:
        """
        Calculates a quality score based on citation count.

        Papers with more citations receive higher scores. Score is normalized between 0.0 and 1.0.

        Args:
            citation_count (int): Number of citations for the paper.
            min_citations (int): Minimum citation count in the dataset.
            max_citations (int): Maximum citation count in the dataset.

        Returns:
            float: Quality score between 0.0 and 1.0 (higher = more citations).
                 Returns 0.0 if citation_count is invalid or negative.

        Raises:
            ValueError: If citation_count is negative (should return 0.0 as fallback).
        """
        if citation_count < 0:
            return 0.0

        if max_citations == min_citations:
            return 1.0 if citation_count > 0 else 0.0

        # Normalize score to [0.0, 1.0]
        score = (citation_count - min_citations) / (
            max_citations - min_citations
        )

        return max(0.0, min(score, 1.0))

    def detect_query_topic(self, query: str) -> Optional[str]:
        """
        Detects the most likely research topic for a query based on keyword matching.

        Topics:
        - 'machine-learning': Keywords like 'machine learning', 'neural network', 'deep learning', 'model', 'training'
        - 'natural-language-processing': Keywords like 'NLP', 'language', 'text', 'transformer', 'BERT', 'GPT'
        - 'computer-vision': Keywords like 'image', 'vision', 'CNN', 'convolutional', 'detection', 'recognition'
        - 'mathematics': Keywords like 'mathematical', 'theorem', 'proof', 'algorithm', 'optimization'
        - 'statistics': Keywords like 'statistical', 'probability', 'distribution', 'regression', 'analysis'

        Args:
            query (str): The user's query string.

        Returns:
            Optional[str]: Detected topic, or None if no clear match.

        Raises:
            TypeError: If query is not a string.
        """
        if not isinstance(query, str):
            raise TypeError("query must be a string")

        query_lower = query.lower()
        topic_keywords = {
            'machine-learning': ['machine learning', 'neural network', 'deep learning', 'model', 'training'],
            'natural-language-processing': ['nlp', 'language', 'text', 'transformer', 'bert', 'gpt'],
            'computer-vision': ['image', 'vision', 'cnn', 'convolutional', 'detection', 'recognition'],
            'mathematics': ['mathematical', 'theorem', 'proof', 'algorithm', 'optimization'],
            'statistics': ['statistical', 'probability', 'distribution', 'regression', 'analysis']
        }

        for topic, keywords in topic_keywords.items():
            if any(keyword in query_lower for keyword in keywords):
                return topic

        return None

    def calculate_topic_score(self, paper_topic: str, detected_topic: Optional[str]) -> float:
        """
        Calculates a topic matching score.

        Args:
            paper_topic (str): The topic of the paper.
            detected_topic (Optional[str]): The detected topic from the query.

        Returns:
            float: 1.0 if topics match or no topic detected, 0.0 otherwise.
        """
        if detected_topic is None:
            return 1.0  # No topic detected, neutral score

        if paper_topic.lower() == detected_topic.lower():
            return 1.0  # Exact match

        return 0.0  # No match

    def combine_scores(
        self,
        similarity_score: float,
        freshness_score: float,
        quality_score: float,
        topic_score: float
    ) -> float:
        """
        Combines multiple signals into a final score using weighted average.

        Args:
            similarity_score (float): The semantic similarity score (0.0 to 1.0).
            freshness_score (float): The freshness score (0.0 to 1.0).
            quality_score (float): The quality score (0.0 to 1.0).
            topic_score (float): The topic match score (0.0 or 1.0).

        Returns:
            float: Combined final score.

        Raises:
            TypeError: If any score is not a float.
        """
        if not all(isinstance(score, (int, float)) for score in [similarity_score, freshness_score, quality_score, topic_score]):
            raise TypeError("All scores must be floats")

        final_score = (
            self.similarity_weight * similarity_score +
            self.freshness_weight * freshness_score +
            self.quality_weight * quality_score +
            self.topic_weight * topic_score
        )
        return final_score

    def rank_documents(
        self,
        df: pd.DataFrame,
        query: str,
        similarity_scores: List[float]
    ) -> pd.DataFrame:
        """
        Filters and ranks papers based on metadata and similarity scores.

        Process:
        1. Detect query topic.
        2. Calculate freshness scores for all papers.
        3. Calculate quality scores for all papers.
        4. Calculate topic match scores.
        5. Combine all scores with weights.
        6. Return ranked DataFrame.

        Args:
            df (pd.DataFrame): DataFrame with retrieved papers containing:
                - 'id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department'
            query (str): The user's query.
            similarity_scores (List[float]): Semantic similarity scores from vector search.

        Returns:
            pd.DataFrame: Ranked DataFrame with added 'freshness_score', 'quality_score',
                          'topic_score', 'similarity_score', and 'final_score' columns,
                          sorted by 'final_score' in descending order.

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            TypeError: If inputs have incorrect types.
        """
        if len(similarity_scores) != len(df):
            raise ValueError(
                "similarity_scores length must match DataFrame length."
            )

        required_columns = ['id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department']

        if df.empty or not all(col in df.columns for col in required_columns):
            raise ValueError("DataFrame is empty or missing required columns")

        if not isinstance(query, str):
            raise TypeError("query must be a string")

        df = df.copy()

        # Add similarity scores BEFORE filtering
        df["similarity_score"] = similarity_scores

        # Detect topic from query
        topic_detected = self.detect_query_topic(query)

        # Calculate topic scores for all documents
        df["topic_score"] = df["topic"].apply(
            lambda t: self.calculate_topic_score(t, topic_detected)
        )

        # Calculate freshness scores
        dates = pd.to_datetime(df["publication_date"])
        min_date = dates.min().to_pydatetime()
        max_date = dates.max().to_pydatetime()

        df["freshness_score"] = df["publication_date"].apply(
            lambda d: self.calculate_freshness_score(
                d,
                min_date,
                max_date,
            )
        )

        # Calculate quality scores
        min_citations = df["citation_count"].min()
        max_citations = df["citation_count"].max()

        df["quality_score"] = df["citation_count"].apply(
            lambda c: self.calculate_quality_score(
                c,
                min_citations,
                max_citations,
            )
        )

        # Combine all scores
        df["final_score"] = df.apply(
            lambda row: self.combine_scores(
                row["similarity_score"],
                row["freshness_score"],
                row["quality_score"],
                row["topic_score"],
            ),
            axis=1,
        )

        return df.sort_values(
            by="final_score", ascending=False
        ).reset_index(drop=True)

filter = MetadataFilter()

scores = [0.9, 0.8, 0.7, 0.6, 0.5]
data = pd.DataFrame({
'id': ['PAPER001', 'PAPER002', 'PAPER003', 'PAPER004', 'PAPER005'],
'title': [
    'Neural Network Architectures for NLP',
    'Attention Mechanisms in Transformers',
    'Computer Vision with CNNs',
    'Mathematical Foundations of Deep Learning',
    'Statistical Methods for Data Analysis'
],
'content': [
    'This paper introduces novel transformer architectures for natural language processing. The model achieves 95% accuracy on benchmark datasets. Published in 2024.',
    'We explore attention mechanisms in transformer models. The research shows significant improvements in language understanding tasks. Published in 2023.',
    'Convolutional neural networks for image recognition. The approach achieves state-of-the-art results on ImageNet with 98% accuracy. Published in 2022.',
    'Mathematical analysis of deep learning optimization. The paper proves convergence theorems for gradient descent algorithms. Published in 2021.',
    'Statistical methods for analyzing large datasets. The paper introduces new regression techniques with applications in machine learning. Published in 2020.'
],
'topic': [
    'machine-learning',
    'natural-language-processing',
    'computer-vision',
    'mathematics',
    'statistics'
],
'publication_date': ['2024-09-15', '2023-06-20', '2022-03-10', '2021-11-05', '2020-08-12'],
'citation_count': [142, 89, 256, 78, 45],
'author_department': [
    'computer-science',
    'computer-science',
    'computer-science',
    'mathematics',
    'statistics'
],
'venue': ['NeurIPS', 'ICML', 'CVPR', 'arXiv', 'JMLR']})

filter.rank_documents(data, "Machine learning", scores)

,id,title,content,topic,publication_date,citation_count,author_department,venue,similarity_score,topic_score,freshness_score,quality_score,final_score
0,PAPER001,Neural Network Architectures for NLP,This paper introduces novel transformer archit...,machine-learning,2024-09-15,142,computer-science,NeurIPS,0.9,1.0,1.000000,0.459716,0.861943
1,PAPER003,Computer Vision with CNNs,Convolutional neural networks for image recogn...,computer-vision,2022-03-10,256,computer-science,CVPR,0.7,0.0,0.384615,1.000000,0.525385
2,PAPER002,Attention Mechanisms in Transformers,We explore attention mechanisms in transformer...,natural-language-processing,2023-06-20,89,computer-science,ICML,0.8,0.0,0.696990,0.208531,0.490803
3,PAPER004,Mathematical Foundations of Deep Learning,Mathematical analysis of deep learning optimiz...,mathematics,2021-11-05,78,mathematics,arXiv,0.6,0.0,0.301003,0.156398,0.301581
4,PAPER005,Statistical Methods for Data Analysis,Statistical methods for analyzing large datase...,statistics,2020-08-12,45,statistics,JMLR,0.5,0.0,0.000000,0.000000,0.150000


In [43]:
"""Answer validation module for Research Paper Library Search System.

This module provides functionality to validate answers generated by a RAG system
for accuracy, completeness, relevance, and hallucination detection. It uses
rule-based validation (no LLM required).
"""

import re
from typing import Dict, List, Set


class AnswerValidator:
    """
    Validates generated answers for accuracy, completeness, and relevance.

    This class evaluates answers generated by RAG systems to ensure they meet quality standards:
    - Accuracy: Answer correctly addresses the query and matches source content
    - Completeness: Answer contains all necessary information
    - Relevance: Answer is directly related to the query
    - Hallucination Detection: Identifies when answer contains unsupported information
    - Keyword Matching: Extracts and matches keywords between query and answer
    - Confidence Scoring: Calculates overall confidence in the answer

    The validator uses rule-based methods (no LLM required).
    """

    def __init__(self):
        """
        Initializes the AnswerValidator.
        """
        pass

    def extract_keywords(self, text: str) -> List[str]:
        """
        Extracts important keywords from a text string.

        Args:
            text (str): The text from which to extract keywords.

        Returns:
            List[str]: List of extracted keywords (lowercase, no duplicates).

        Raises:
            TypeError: If text is None or not a string.
        """
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        # Simple keyword extraction: split by non-word characters, filter out short words
        words = re.findall(r'\b\w{4,}\b', text.lower())
        keywords = sorted(set(words))  # Remove duplicates and sort
        return keywords

    def check_accuracy(self, answer: str, source_documents: List[str]) -> Dict:
        """
        Checks if the answer accurately represents information from source documents.

        Validates:
        - Numbers in answer appear in source documents
        - Dates in answer appear in source documents
        - Key claims are supported by source content

        Args:
            answer (str): The generated answer to validate.
            source_documents (List[str]): List of source document texts.

        Returns:
            dict: Accuracy results containing:
                - 'is_accurate' (bool): Whether answer is accurate
                - 'unsupported_numbers' (List[str]): Numbers in answer not found in sources
                - 'unsupported_dates' (List[str]): Dates in answer not found in sources
                - 'supported_claims' (int): Number of claims supported by sources

        Raises:
            TypeError: If answer is None or not a string.
            ValueError: If answer is empty or source_documents is empty.
        """
        if  not isinstance(answer, str):
            raise TypeError("answer must be a non-empty string")

        if not source_documents or not all(isinstance(doc, str) for doc in source_documents):
            raise ValueError("source_documents must be a list of non-empty strings")

        if not answer.strip():
            raise ValueError("answer cannot be empty")

        numbers_in_answer = re.findall(r'\b\d+\b', answer)
        dates_in_answer = re.findall(r'\b\d{4}-\d{2}-\d{2}\b', answer)
        numbers_in_sources = set()
        dates_in_sources = set()

        for doc in source_documents:
            numbers_in_sources.update(re.findall(r'\b\d+\b', doc))
            dates_in_sources.update(re.findall(r'\b\d{4}-\d{2}-\d{2}\b', doc))

        # Check for unsupported numbers
        unsupported_numbers = [num for num in numbers_in_answer if num not in numbers_in_sources]
        unsupported_dates = [date for date in dates_in_answer if date not in dates_in_sources]

        # Check for supported claims (simple keyword matching)
        source_keywords = set(
            self.extract_keywords(" ".join(source_documents))
        )

        supported_claims = sum(
            1
            for claim in self.extract_keywords(answer)
            if claim in source_keywords
        )

        return {
            "is_accurate": not unsupported_numbers and not unsupported_dates and supported_claims > 0,
            "unsupported_numbers": unsupported_numbers,
            "unsupported_dates": unsupported_dates,
            "supported_claims": supported_claims
        }

    def assess_completeness(self, query: str, answer: str) -> Dict:
        """
        Assesses whether the answer addresses the full query.

        Checks:
        - Answer length (too short may indicate incompleteness)
        - Keyword overlap between query and answer
        - Presence of vague language indicating uncertainty

        Args:
            query (str): The original user query.
            answer (str): The generated answer.

        Returns:
            dict: Completeness results containing:
                - 'is_complete' (bool): Whether answer is complete
                - 'keyword_overlap' (List[str]): Common keywords between query and answer
                - 'has_vague_language' (bool): Whether answer contains vague language
                - 'completeness_score' (float): Score from 0.0 to 1.0

        Raises:
            TypeError: If query or answer is None or not a string.
            ValueError: If query or answer is empty.
        """
        if not isinstance(query, str):
            raise TypeError("query must be a non-empty string")

        if not isinstance(answer, str):
            raise TypeError("answer must be a non-empty string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        if not answer.strip():
            raise ValueError("answer cannot be empty")

        if len(answer.split()) < 10:  # Arbitrary threshold for completeness
            is_complete = False
        else:
            is_complete = True

        query_keywords = set(self.extract_keywords(query))
        answer_keywords = set(self.extract_keywords(answer))

        keyword_overlap = sorted(query_keywords & answer_keywords)

        has_vague_language = any(word in answer.lower() for word in ["maybe", "might", "could", "possibly"])

        return {
            "is_complete": is_complete,
            "keyword_overlap": keyword_overlap,
            "has_vague_language": has_vague_language,
            "completeness_score": 1.0 if is_complete else 0.0
        }

    def detect_hallucinations(self, answer: str, source_documents: List[str]) -> Dict:
        """
        Detects hallucinations (unsupported claims) in the answer.

        Detects:
        - Numbers that don't appear in source documents
        - Dates that don't appear in source documents
        - Specific claims (names, technical terms) not found in sources

        Args:
            answer (str): The generated answer to check.
            source_documents (List[str]): List of source document texts.

        Returns:
            dict: Hallucination detection results containing:
                - 'has_hallucinations' (bool): Whether hallucinations were detected
                - 'hallucinated_numbers' (List[str]): Numbers not found in sources
                - 'hallucinated_dates' (List[str]): Dates not found in sources
                - 'hallucination_score' (float): Score from 0.0 to 1.0 (higher = more hallucinations)

        Raises:
            TypeError: If answer is None or not a string.
            ValueError: If answer is empty or source_documents is empty.
        """
        if not isinstance(answer, str):
            raise TypeError("answer must be a non-empty string")

        if not source_documents or not all(isinstance(doc, str) for doc in source_documents):
            raise ValueError("source_documents must be a list of non-empty strings")

        if not answer.strip():
            raise ValueError("answer cannot be empty")

        numbers_in_answer = re.findall(r'\b\d+\b', answer)
        dates_in_answer = re.findall(r'\b\d{4}-\d{2}-\d{2}\b', answer)
        numbers_in_sources = set()
        dates_in_sources = set()

        for doc in source_documents:
            numbers_in_sources.update(re.findall(r'\b\d+\b', doc))
            dates_in_sources.update(re.findall(r'\b\d{4}-\d{2}-\d{2}\b', doc))

        hallucinated_numbers = [num for num in numbers_in_answer if num not in numbers_in_sources]
        hallucinated_dates = [date for date in dates_in_answer if date not in dates_in_sources]

        has_hallucinations = bool(hallucinated_numbers or hallucinated_dates)

        answer_keywords = set(self.extract_keywords(answer))
        source_keywords = set(
            self.extract_keywords(" ".join(source_documents))
        )

        hallucinated_claims = list(answer_keywords - source_keywords)

        hallucination_score = (
            len(hallucinated_numbers)
            + len(hallucinated_dates)
            + len(hallucinated_claims)
        ) / max(
            1,
            len(answer_keywords)
            + len(numbers_in_answer)
            + len(dates_in_answer)
        )

        return {
            "has_hallucinations": has_hallucinations,
            "hallucinated_numbers": hallucinated_numbers,
            "hallucinated_dates": hallucinated_dates,
            "hallucination_score": hallucination_score
        }

    def calculate_confidence(self, validation_results: Dict) -> float:
        """
        Calculates overall confidence score for the answer.

        Combines:
        - Accuracy score
        - Completeness score
        - Hallucination score (inverted)

        Args:
            validation_results (dict): Results from validate_answer() containing:
                - 'is_accurate' (bool)
                - 'is_complete' (bool)
                - 'has_hallucinations' (bool)
                - 'relevance_score' (float)
                - 'completeness_score' (float)
                - 'hallucination_score' (float)

        Returns:
            float: Confidence score from 0.0 to 1.0 (higher = more confident).

        Raises:
            ValueError: If validation_results is missing required keys.
        """
        required_keys = [
            'is_accurate',
            'is_complete',
            'has_hallucinations',
            'relevance_score',
            'completeness_score',
            'hallucination_score'
        ]
        if not all(key in validation_results for key in required_keys):
            raise ValueError(f"validation_results is missing required keys: {required_keys}")

        accuracy_score = 1.0 if validation_results['is_accurate'] else 0.0
        completeness_score = validation_results['completeness_score']
        hallucination_score = 1.0 - validation_results['hallucination_score']

        confidence_score = (
            accuracy_score
            + validation_results["relevance_score"]
            + completeness_score
            + hallucination_score
        ) / 4.0

        return confidence_score

    def validate_answer(
        self,
        query: str,
        answer: str,
        source_documents: List[str],
    ) -> Dict:
        """
        Validates a generated answer using accuracy, completeness,
        relevance, and hallucination detection.

        Args:
            query (str): Original user query.
            answer (str): Generated answer.
            source_documents (List[str]): Source document texts.

        Returns:
            Dict: Validation results including confidence score.

        Raises:
            TypeError: If inputs have invalid types.
            ValueError: If inputs are empty.
        """
        # Validate query
        if not isinstance(query, str):
            raise TypeError("query must be a string.")

        if not query.strip():
            raise ValueError("query cannot be empty.")

        # Validate answer
        if not isinstance(answer, str):
            raise TypeError("answer must be a string.")

        if not answer.strip():
            raise ValueError("answer cannot be empty.")

        # Validate source documents
        if not isinstance(source_documents, list):
            raise TypeError("source_documents must be a list.")

        if not source_documents:
            raise ValueError("source_documents cannot be empty.")

        if not all(isinstance(doc, str) for doc in source_documents):
            raise TypeError("Every source document must be a string.")

        # Run individual validations
        accuracy_results = self.check_accuracy(answer, source_documents)
        completeness_results = self.assess_completeness(query, answer)
        hallucination_results = self.detect_hallucinations(
            answer,
            source_documents,
        )

        # Compute relevance score from keyword overlap
        query_keywords = set(self.extract_keywords(query))

        if query_keywords:
            relevance_score = (
                len(completeness_results["keyword_overlap"])
                / len(query_keywords)
            )
        else:
            relevance_score = 0.0

        validation_results = {
            "is_accurate": accuracy_results["is_accurate"],
            "is_complete": completeness_results["is_complete"],
            "has_hallucinations": hallucination_results["has_hallucinations"],
            "relevance_score": relevance_score,
            "completeness_score": completeness_results["completeness_score"],
            "hallucination_score": hallucination_results["hallucination_score"],
            "keyword_overlap": completeness_results["keyword_overlap"],
            "unsupported_numbers": accuracy_results["unsupported_numbers"],
            "unsupported_dates": accuracy_results["unsupported_dates"],
            "supported_claims": accuracy_results["supported_claims"],
            "hallucinated_numbers": hallucination_results["hallucinated_numbers"],
            "hallucinated_dates": hallucination_results["hallucinated_dates"],
        }

        validation_results["confidence_score"] = self.calculate_confidence(
            validation_results
        )

        return validation_results

In [44]:
"""Vector store module for semantic search in Research Paper Library Search System.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from research paper content using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique paper identifier.
    - 'title': The paper title.
    - 'content': The paper content (used for embeddings).
    - 'topic': Research topic.
    - 'publication_date': Publication date.
    - 'citation_count': Citation count.
    - 'author_department': Author department.
    - 'venue': Publication venue (optional).

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.
    Embeddings should be generated from paper content (title + content).

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    Implements brute-force approach: compute similarity with all vectors, then return top-k.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
                Default: "all-MiniLM-L6-v2" (384 dimensions)
        """
        self.model_name = model_name
        self.model = None

    def _ensure_model_loaded(self):
        """Lazy load the SentenceTransformer model."""
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., paper titles + content).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).
                For all-MiniLM-L6-v2, embedding_dim = 384.

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        # Validate input
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        if len(texts) == 0:
            raise ValueError("texts list cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embeddings
        embeddings = self.model.encode(texts, convert_to_numpy=True)

        return embeddings

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for paper content (title + content).
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing research papers with required fields:
                - 'id': Unique paper ID.
                - 'title': Paper title.
                - 'content': Paper content.
                - 'topic': Research topic.
                - 'publication_date': Publication date.
                - 'citation_count': Citation count.
                - 'author_department': Author department.
                - 'venue': Publication venue (optional).
            index_file_name (str): Name of the index file (e.g., 'paper_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing:
                - 'embeddings': numpy array of embeddings (shape: [n_docs, 384])
                - 'id': list of paper IDs
                - 'title': list of titles
                - 'content': list of content
                - 'topic': list of topics
                - 'publication_date': list of publication dates
                - 'citation_count': list of citation counts
                - 'author_department': list of departments
                - 'venue': list of venues (may contain None values)

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        # Validate DataFrame
        if not isinstance(df, pd.DataFrame):
            raise ValueError("df must be a pandas DataFrame")

        if len(df) == 0:
            raise ValueError("DataFrame is empty")

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        # Generate embeddings from paper content (title + content)
        texts = [f"{row['title']}\n\n{row['content']}" for _, row in df.iterrows()]
        embeddings = self.generate_embeddings(texts)

        # Create index dictionary
        index = {
            'embeddings': embeddings,
            'id': df['id'].tolist(),
            'title': df['title'].tolist(),
            'content': df['content'].tolist(),
            'topic': df['topic'].tolist(),
            'publication_date': df['publication_date'].tolist(),
            'citation_count': df['citation_count'].tolist(),
            'author_department': df['author_department'].tolist()
        }

        # Add venue if present
        if 'venue' in df.columns:
            index['venue'] = df['venue'].tolist()
        else:
            index['venue'] = [None] * len(df)

        # Create directory if it doesn't exist
        index_folder = Path(index_folder_name)
        index_folder.mkdir(parents=True, exist_ok=True)

        # Save index to disk
        index_path = index_folder / index_file_name
        with open(index_path, 'wb') as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'paper_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata (same structure as create_index).

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        # Construct full path
        index_path = Path(index_folder_name) / index_file_name

        # Check if file exists
        if not index_path.exists():
            raise FileNotFoundError(f"Index file not found: {index_path}")

        # Load index
        try:
            with open(index_path, 'rb') as f:
                index = pickle.load(f)
        except Exception as e:
            raise ValueError(f"Failed to load index: {str(e)}")

        # Validate index structure
        required_keys = ['embeddings', 'id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department']
        missing_keys = [key for key in required_keys if key not in index]
        if missing_keys:
            raise KeyError(f"Index missing required keys: {missing_keys}")

        # Validate embeddings shape
        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        if len(index['embeddings']) == 0:
            raise ValueError("Index embeddings array is empty")

        # Validate metadata lengths match
        n_docs = len(index['embeddings'])
        for key in required_keys:
            if len(index[key]) != n_docs:
                raise ValueError(f"Index metadata length mismatch: {key} has {len(index[key])} items, expected {n_docs}")

        return index

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).
                For all-MiniLM-L6-v2, shape = (384,).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        # Validate input
        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embedding (single text, returns 1D array)
        embedding = self.model.encode(query, convert_to_numpy=True)

        # Ensure it's 1D
        if embedding.ndim > 1:
            embedding = embedding[0]

        return embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar entries from the index using cosine similarity.

        Implementation uses brute-force approach:
        1. Compute cosine similarity between query_embedding and all embeddings in index
        2. Sort results by similarity score (descending)
        3. Return top-k matches

        Cosine similarity formula:
        similarity = dot(a, b) / (norm(a) * norm(b))

        Args:
            query_embedding (np.ndarray): The embedding of the input query (shape: [384]).
            index (dict): The index containing document embeddings and metadata.
                Expected structure: {'embeddings': np.ndarray, 'id': list, ...}
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.
                  - document_index: Index position in the original DataFrame/index
                  - similarity_score: Cosine similarity score (float, typically 0.0 to 1.0)

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        # Validate inputs
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("query_embedding must be a numpy array")

        if 'embeddings' not in index:
            raise ValueError("Index must contain 'embeddings' key")

        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        # Get document embeddings
        doc_embeddings = index['embeddings']

        # Validate shapes
        if query_embedding.ndim != 1:
            raise ValueError("query_embedding must be 1D array")

        if doc_embeddings.ndim != 2:
            raise ValueError("doc_embeddings must be 2D array")

        if query_embedding.shape[0] != doc_embeddings.shape[1]:
            raise ValueError(f"Dimension mismatch: query has {query_embedding.shape[0]} dims, docs have {doc_embeddings.shape[1]} dims")

        # Compute cosine similarity for all documents (brute-force)
        # Cosine similarity = dot(a, b) / (norm(a) * norm(b))
        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(doc_embeddings, axis=1)

        # Compute dot products
        dot_products = np.dot(doc_embeddings, query_embedding)

        # Compute cosine similarities
        similarities = dot_products / (query_norm * doc_norms)

        # Handle any NaN values (shouldn't happen, but safety check)
        similarities = np.nan_to_num(similarities, nan=0.0)

        # Get top-k indices
        top_k_indices = np.argsort(similarities)[::-1][:k]

        # Create list of (index, similarity) tuples
        results = [(int(idx), float(similarities[idx])) for idx in top_k_indices]

        return results



In [45]:
"""Document loading module for Research Paper Library Search System.

This module provides functionality to load research paper data from JSON files
and convert it into formats suitable for RAG processing, including pandas
DataFrames and LangChain Document objects.
"""

import json
from pathlib import Path
from typing import List

import pandas as pd
from langchain_core.documents import Document


class DocumentLoader:
    """
    A utility class for loading and converting JSON research paper data into LangChain Document objects.

    Input Requirement:
    - The input must be a JSON file with a list of research paper entries.
    - Each entry must contain the following fields:
        - 'id': Unique identifier for each paper.
        - 'title': The paper title.
        - 'content': The abstract and key findings.
        - 'topic': Research topic (e.g., 'machine-learning', 'natural-language-processing', 'computer-vision', 'mathematics', 'statistics').
        - 'publication_date': Date when paper was published (YYYY-MM-DD format).
        - 'citation_count': Number of citations (integer).
        - 'author_department': Department (e.g., 'computer-science', 'mathematics', 'statistics').
        - 'venue': Publication venue (optional, e.g., 'NeurIPS', 'ICML', 'arXiv').

    Example JSON format:
    [
        {
            "id": "PAPER001",
            "title": "Neural Network Architectures for NLP",
            "content": "This paper introduces novel transformer architectures...",
            "topic": "machine-learning",
            "publication_date": "2024-09-15",
            "citation_count": 142,
            "author_department": "computer-science",
            "venue": "NeurIPS"
        }
    ]
    """

    def __init__(self, json_file: str):
        """
        Initializes the DocumentLoader with the path to a JSON file.

        Args:
            json_file (str): Absolute or relative path to the JSON file.

        Raises:
            ValueError: If input is not a non-empty string.
        """
        if not isinstance(json_file, str) or not json_file.strip():
            raise ValueError("json_file must be a non-empty string")
        self.json_file = json_file

    def load_data(self) -> pd.DataFrame:
        """
        Loads and parses research paper data from the specified JSON file.

        Functionality:
        - Verifies that the file exists and is accessible.
        - Parses the JSON content into a pandas DataFrame.
        - Validates that each entry contains all required fields.
        - Returns a DataFrame with all paper data.

        Returns:
            pd.DataFrame: A DataFrame containing all research papers with required columns:
                - 'id': Paper identifier
                - 'title': Paper title
                - 'content': Paper content (abstract and key findings)
                - 'topic': Research topic
                - 'publication_date': Publication date
                - 'citation_count': Citation count
                - 'author_department': Author department
                - 'venue': Publication venue (optional)

        Raises:
            FileNotFoundError: If the file path does not exist.
            json.JSONDecodeError: If file contains invalid JSON.
            ValueError: If required fields are missing or DataFrame is empty.
            KeyError: If required columns are not present in the data.
        """
        # Check if file exists
        file_path = Path(self.json_file)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {self.json_file}")

        # Load JSON
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            raise json.JSONDecodeError(f"Invalid JSON in file: {e.msg}", e.doc, e.pos)

        # Validate data is a list
        if not isinstance(data, list):
            raise ValueError("JSON file must contain a list of entries")

        # Validate data is not empty
        if len(data) == 0:
            raise ValueError("JSON file contains no entries")

        # Convert to DataFrame
        df = pd.DataFrame(data)

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise KeyError(f"Missing required columns: {missing_columns}")

        # Validate no missing values in required fields
        for col in required_columns:
            if df[col].isna().any():
                raise ValueError(f"Column '{col}' contains missing values")

        # Validate DataFrame is not empty after cleaning
        if len(df) == 0:
            raise ValueError("DataFrame is empty after validation")

        return df

    def create_documents(self, data: pd.DataFrame) -> List[Document]:
        """
        Converts validated paper data from a DataFrame into LangChain Document objects.

        Args:
            data (pandas.DataFrame): A DataFrame where each row represents a research paper. Required columns:
                - 'id'
                - 'title'
                - 'content'
                - 'topic'
                - 'publication_date'
                - 'citation_count'
                - 'author_department'
                - 'venue' (optional)

        Returns:
            List[Document]: A list of LangChain-compatible Document objects with metadata.
                Each Document should have:
                - page_content: The paper title + content
                - metadata: Dictionary containing 'id', 'title', 'topic', 'publication_date', 'citation_count', 'author_department', 'venue'

        Raises:
            ValueError: If required columns are missing or DataFrame is empty.
            TypeError: If input is not a pandas DataFrame.
        """
        # Validate input type
        if not isinstance(data, pd.DataFrame):
            raise TypeError("Input must be a pandas DataFrame")

        # Validate DataFrame is not empty
        if len(data) == 0:
            raise ValueError("DataFrame is empty")

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'topic', 'publication_date', 'citation_count', 'author_department']
        missing_columns = [col for col in required_columns if col not in data.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        # Create Document objects
        documents = []
        for _, row in data.iterrows():
            # Use title + content as page_content
            page_content = f"{row['title']}\n\n{row['content']}"

            # Create metadata dictionary
            metadata = {
                'id': row['id'],
                'title': row['title'],
                'topic': row['topic'],
                'publication_date': row['publication_date'],
                'citation_count': int(row['citation_count']),
                'author_department': row['author_department']
            }

            # Add venue if present
            if 'venue' in row and pd.notna(row['venue']):
                metadata['venue'] = row['venue']

            # Create LangChain Document
            doc = Document(page_content=page_content, metadata=metadata)
            documents.append(doc)

        return documents



In [46]:
"""Research Paper Library Search System.

This module integrates VectorStore, MetadataFilter, and AnswerValidator into a
complete search system for research papers with production-grade features.
"""

from typing import Dict, List, Optional

import pandas as pd

# from src.document_loader import DocumentLoader
# from src.vector_store import VectorStore
# from src.metadata_filter import MetadataFilter
# from src.answer_validator import AnswerValidator


class ResearchSearchSystem:
    """
    A complete research paper search system with metadata filtering and answer validation.

    This system integrates:
    - VectorStore: For semantic similarity search
    - MetadataFilter: For filtering and ranking by metadata
    - AnswerValidator: For validating answer quality

    The system processes queries, retrieves relevant papers, filters and ranks them,
    generates answers, and validates the results.
    """

    def __init__(
        self,
        vector_store: VectorStore,
        metadata_filter: MetadataFilter,
        answer_validator: AnswerValidator,
        index: Dict
    ):
        """
        Initializes the ResearchSearchSystem with required components.

        Args:
            vector_store (VectorStore): VectorStore instance for similarity search.
            metadata_filter (MetadataFilter): MetadataFilter instance for filtering/ranking.
            answer_validator (AnswerValidator): AnswerValidator instance for validation.
            index (Dict): Pre-loaded vector index containing embeddings and metadata.
        """
        self.vector_store = vector_store
        self.metadata_filter = metadata_filter
        self.answer_validator = answer_validator
        self.index = index

    def search(
        self,
        query: str,
        k: int = 5,
        topic: Optional[str] = None,
        author_department: Optional[str] = None,
    ) -> Dict:
        """
        Searches the research paper index and returns validated results.
        """
        # Validate inputs
        if not isinstance(query, str):
            raise TypeError("query must be a string.")

        if not query.strip():
            raise ValueError("query cannot be empty.")

        if not isinstance(k, int):
            raise TypeError("k must be an integer.")

        if k <= 0:
            raise ValueError("k must be greater than zero.")

        # Generate query embedding
        query_embedding = self.vector_store.get_query_embedding(query)

        # Retrieve top-k matches
        matches = self.vector_store.find_top_k_matches(
            query_embedding,
            self.index,
            k,
        )

        if not matches:
            return {
                "answer": "No relevant papers found for your query.",
                "confidence_score": 0.0,
                "source_papers": [],
                "validation_results": {},
                "top_papers": [],
            }

        # Build DataFrame from retrieved papers
        rows = []

        for doc_idx, similarity in matches:
            rows.append({
                "id": self.index["id"][doc_idx],
                "title": self.index["title"][doc_idx],
                "content": self.index["content"][doc_idx],
                "topic": self.index["topic"][doc_idx],
                "author_department": self.index["author_department"][doc_idx],
                "publication_date": self.index["publication_date"][doc_idx],
                "citation_count": self.index["citation_count"][doc_idx],
                "similarity_score": similarity,
            })

        papers_df = pd.DataFrame(rows)

        # Extract similarity scores BEFORE any filtering
        similarity_scores = papers_df["similarity_score"].tolist()

        # Rank papers (rank_documents signature is: df, query, similarity_scores)
        ranked_df = self.metadata_filter.rank_documents(
            papers_df,
            query,
            similarity_scores,
        )

        # Apply explicit metadata filters AFTER ranking (only if user provides them)
        if topic is not None or author_department is not None:
            try:
                ranked_df = self._apply_filters(
                    ranked_df,
                    topic,
                    author_department,
                )
            except ValueError:
                # If filtering results in no matches, return empty result
                return {
                    "answer": "No relevant papers found for your query.",
                    "confidence_score": 0.0,
                    "source_papers": [],
                    "validation_results": {},
                    "top_papers": [],
                }

        # Check if we have any results
        if ranked_df.empty:
            return {
                "answer": "No relevant papers found for your query.",
                "confidence_score": 0.0,
                "source_papers": [],
                "validation_results": {},
                "top_papers": [],
            }

        # Generate answer
        answer = self._generate_answer(
            query,
            ranked_df.to_dict("records"),
        )

        # Validate answer
        validation_results = self.answer_validator.validate_answer(
            query=query,
            answer=answer,
            source_documents=ranked_df["content"].tolist(),
        )

        return self._format_response(
            answer,
            validation_results,
            ranked_df.to_dict("records"),
        )

    def _apply_filters(
        self,
        df: pd.DataFrame,
        topic: Optional[str] = None,
        author_department: Optional[str] = None
    ) -> pd.DataFrame:
        """
        Applies metadata filters to the DataFrame.

        Args:
            df (pd.DataFrame): DataFrame with paper data.
            topic (Optional[str]): Topic to filter by.
            author_department (Optional[str]): Department to filter by.

        Returns:
            pd.DataFrame: Filtered DataFrame.
        """
        return self.metadata_filter.filter_by_metadata(df, topic, author_department)

    def _generate_answer(
        self,
        query: str,
        top_papers: List[Dict],
    ) -> str:
        """
        Generates a simple extractive answer from retrieved papers.
        """
        if not top_papers:
            return "No relevant information found."

        sections = []

        for paper in top_papers:
            title = paper.get("title", "Untitled")
            content = paper.get("content", "")

            sections.append(
                f"{title}\n{content[:200]}"
            )

        return "\n\n".join(sections)

    def _format_response(
        self,
        answer: str,
        validation_results: Dict,
        top_papers: List[Dict]
    ) -> Dict:
        """
        Formats the final response with answer, confidence, and source papers.

        Args:
            answer (str): Generated answer.
            validation_results (Dict): Validation results from AnswerValidator.
            top_papers (List[Dict]): Top-ranked papers with scores.

        Returns:
            dict: Formatted response dictionary.
        """
        return {
            "answer": answer,
            "confidence_score": validation_results.get("confidence_score", 0.0),
            "source_papers": [paper for paper in top_papers],
            "validation_results": validation_results,
            "top_papers": top_papers
        }

In [47]:
# Test the ResearchSearchSystem integration

# Create test data
test_data = pd.DataFrame({
    'id': ['PAPER001', 'PAPER002', 'PAPER003', 'PAPER004', 'PAPER005'],
    'title': [
        'Neural Network Architectures for NLP',
        'Attention Mechanisms in Transformers',
        'Computer Vision with CNNs',
        'Mathematical Foundations of Deep Learning',
        'Statistical Methods for Data Analysis'
    ],
    'content': [
        'This paper introduces novel transformer architectures for natural language processing. The model achieves 95% accuracy on benchmark datasets.',
        'We explore attention mechanisms in transformer models. The research shows significant improvements in language understanding tasks.',
        'Convolutional neural networks for image recognition. The approach achieves state-of-the-art results on ImageNet with 98% accuracy.',
        'Mathematical analysis of deep learning optimization. The paper proves convergence theorems for gradient descent algorithms.',
        'Statistical methods for analyzing large datasets. The paper introduces new regression techniques with applications in machine learning.'
    ],
    'topic': [
        'machine-learning',
        'natural-language-processing',
        'computer-vision',
        'mathematics',
        'statistics'
    ],
    'publication_date': ['2024-09-15', '2023-06-20', '2022-03-10', '2021-11-05', '2020-08-12'],
    'citation_count': [142, 89, 256, 78, 45],
    'author_department': [
        'computer-science',
        'computer-science',
        'computer-science',
        'mathematics',
        'statistics'
    ],
    'venue': ['NeurIPS', 'ICML', 'CVPR', 'arXiv', 'JMLR']
})

# Initialize components
vector_store = VectorStore()
metadata_filter = MetadataFilter()
answer_validator = AnswerValidator()

# Create index
index = vector_store.create_index(
    test_data,
    'test_index.pkl',
    'test_indexes'
)

# Create search system
search_system = ResearchSearchSystem(
    vector_store=vector_store,
    metadata_filter=metadata_filter,
    answer_validator=answer_validator,
    index=index
)

# Test search
result = search_system.search("machine learning neural networks", k=3)

print("Search Results:")
print(f"Confidence Score: {result['confidence_score']:.2f}")
print(f"\nTop Papers:")
for i, paper in enumerate(result['top_papers'][:3], 1):
    print(f"\n{i}. {paper['title']}")
    print(f"   Topic: {paper['topic']}")
    print(f"   Final Score: {paper.get('final_score', 'N/A'):.3f}")
    print(f"   Similarity: {paper['similarity_score']:.3f}")

print(f"\nAnswer Preview:")
print(result['answer'][:300] + "...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5828.33it/s]



Search Results:
Confidence Score: 0.97

Top Papers:

1. Computer Vision with CNNs
   Topic: computer-vision
   Final Score: 0.609
   Similarity: 0.363

2. Mathematical Foundations of Deep Learning
   Topic: mathematics
   Final Score: 0.382
   Similarity: 0.387

3. Statistical Methods for Data Analysis
   Topic: statistics
   Final Score: 0.105
   Similarity: 0.351

Answer Preview:
Computer Vision with CNNs
Convolutional neural networks for image recognition. The approach achieves state-of-the-art results on ImageNet with 98% accuracy.

Mathematical Foundations of Deep Learning
Mathematical analysis of deep learning optimization. The paper proves convergence theorems for gradi...
